# 03 — Nonlinear learning and model design

A nonlinear model allows the thermodynamic response itself to vary across the domain:

\[
\mathbf g=f(T',P').
\]

That flexibility is useful only if it can be controlled with few states. This notebook expands the single **Architecture search** slide into the operational model-design steps that were intentionally compressed in the presentation.

In [ ]:
#@title 0. Workshop setup — run once { display-mode: "form" }
# This cell intentionally hides infrastructure so workshop time stays focused on physics.

from pathlib import Path
import hashlib, importlib.util, os, shutil, subprocess, sys, urllib.request, zipfile

ASSET_URL = "" #@param {type:"string"}
EXPECTED_ASSET_SHA256 = "2e75fd65ad39a9dec41f7b089c2aafe51a6bb14eeee049b055be8ea3953a7bea"
WORKSHOP_ROOT = Path("/content/ThermoRDF-Workshop")
ASSET_NAME = "ThermoRDF-Colab-Assets.zip"

# Local/instructor execution override used only for automated testing.
_local_root = os.environ.get("THERMORDF_WORKSHOP_ROOT", "").strip()
if _local_root:
    WORKSHOP_ROOT = Path(_local_root).resolve()
else:
    ready = (WORKSHOP_ROOT / "data/teaching/stage_02_b48_train_40.csv.gz").is_file()
    if not ready:
        archive = Path("/content") / ASSET_NAME
        if ASSET_URL.strip():
            print("Downloading workshop assets ...")
            urllib.request.urlretrieve(ASSET_URL.strip(), archive)
        else:
            try:
                from google.colab import files
            except ImportError as exc:
                raise RuntimeError("This notebook is configured for Google Colab. Set THERMORDF_WORKSHOP_ROOT for local testing.") from exc
            print(f"Upload the companion file: {ASSET_NAME}")
            uploaded = files.upload()
            if ASSET_NAME not in uploaded:
                raise RuntimeError(f"Expected {ASSET_NAME}. Please rerun this cell and upload that file.")
            archive.write_bytes(uploaded[ASSET_NAME])

        digest = hashlib.sha256(archive.read_bytes()).hexdigest()
        if digest != EXPECTED_ASSET_SHA256:
            raise RuntimeError("Asset bundle checksum mismatch. Use the bundle distributed with these notebooks.")

        if WORKSHOP_ROOT.exists():
            shutil.rmtree(WORKSHOP_ROOT)
        WORKSHOP_ROOT.mkdir(parents=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(WORKSHOP_ROOT)

_required = {"numpy":"numpy", "pandas":"pandas", "matplotlib":"matplotlib", "scikit-learn":"sklearn", "torch":"torch"}
_missing = [pkg for pkg, module in _required.items() if importlib.util.find_spec(module) is None]
if _missing:
    print("Installing missing Colab packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

sys.path.insert(0, str(WORKSHOP_ROOT / "src"))
from thermordf_workshop import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.set_num_threads(min(2, os.cpu_count() or 1))
data = load_workshop_data(WORKSHOP_ROOT)
print(f"Workshop ready | {len(data.train)} training + {len(data.validation)} validation states | {len(data.r_nm)} RDF coordinates")

## Start from a reference nonlinear baseline

The inherited baseline uses three hidden layers of width 128. It is a **reference model, not an optimum**: we first ask what a generic nonlinear network can already do before adapting the design to a low-data problem.

In [ ]:
baseline_prediction = load_prediction_artifact(data, "baseline_mlp")
baseline_result = evaluate_predictions(data, baseline_prediction)
print("baseline architecture: 2 → 128 → 128 → 128 → 725")
print(f"development validation RDF RMSE: {baseline_result.attrs['global_rdf_rmse']:.4f}")

In [ ]:
plot_reference_prediction(
    data, baseline_prediction, "T200_Bar0001",
    label="baseline MLP", line_color="#E98232", with_error=True
);

## Architecture search — change the representation, not the physics

To keep workshop time focused on interpretation, the scientific pipeline has already trained the 12 candidates. Here we execute the **selection logic** over those controlled runs.

Only hidden width and depth change:

\[
h\in\{16,32,64,128\},\qquad L\in\{1,2,3\}.
\]

The data partition, preprocessing, optimiser settings and validation criterion remain fixed.

In [ ]:
architecture = architecture_search(data)
plot_architecture_search(architecture);

In [ ]:
a = architecture.selected
print(f"selected width       : {int(a['width'])}")
print(f"selected depth       : {int(a['depth'])}")
print(f"trainable parameters : {int(a['parameter_count']):,}")
print(f"best validation RMSE : {a['best_validation_rdf_rmse']:.4f}")

### Observe

Increasing width or depth does **not** improve validation performance monotonically. The selected representation is compact rather than maximal.

The presentation groups the rest of the optimisation into the same *Architecture search* story. We now complete it explicitly.

## Hyperparameter search — optimise the selected representation

With \(h=16\) and \(L=2\) fixed, compare a small \(3\times3\) grid of learning rates and weight decays. The thermodynamic data and network architecture do not change.

In [ ]:
hyper = hyperparameter_search(data, architecture)
plot_hyperparameter_search(hyper);

In [ ]:
h = hyper.selected
print(f"learning rate  : {h['learning_rate']:g}")
print(f"weight decay   : {h['weight_decay']:g}")
print(f"best epoch     : {int(h['best_epoch'])}")
print(f"validation RMSE: {h['best_validation_rdf_rmse']:.4f}")

## Execute the selected network

We now load the frozen selected network and perform **fresh inference** at the eight development-validation coordinates. The network itself generates the RDFs; we are not simply displaying a stored comparison.

In [ ]:
selected_mlp = load_frozen_mlp(data)
selected_prediction = selected_mlp.predict_frame(data.validation)
selected_result = evaluate_predictions(data, selected_prediction)
print(f"selected architecture : 2 → 16 → 16 → 725")
print(f"development RDF RMSE  : {selected_result.attrs['global_rdf_rmse']:.4f}")

## Return to exactly the same physical state

Nothing about \(T=200\) K and \(P=1\) bar has changed. Only the model design changed. This is why the visual comparison with the baseline is scientifically meaningful.

In [ ]:
plot_reference_prediction(
    data, selected_prediction, "T200_Bar0001",
    label="selected MLP", line_color="#E98232", with_error=True
);

In [ ]:
baseline_state = reference_state_metrics(data, baseline_prediction, "T200_Bar0001")
selected_state = reference_state_metrics(data, selected_prediction, "T200_Bar0001")
pd.DataFrame({
    "model": ["baseline MLP", "selected MLP"],
    "reference-state RDF RMSE": [baseline_state["rdf_rmse"], selected_state["rdf_rmse"]],
    "reference-state RDF MAE": [baseline_state["rdf_mae"], selected_state["rdf_mae"]],
})

### Interpret

The model-design sequence is complete:

\[
\text{baseline}\rightarrow\text{architecture selection}\rightarrow\text{hyperparameter selection}\rightarrow\text{selected MLP}.
\]

The eight validation states have participated in architecture choice, hyperparameter choice and stopping. Their result is therefore a **development result**, not an untouched final test.

**Next:** freeze the model and change the scientific question from optimisation to physical interrogation.